# Prova - Classificador supervisionado

**Repositório GitHub:** `classificador_projeto` (nome exigido no enunciado; substitua o link após criar o repo: `https://github.com/seu-usuario/classificador_projeto`).


### 1) Qual a base escolhida?

**League of Legends Diamond Ranked Games (10 min)** — dataset público do Kaggle ([link](https://www.kaggle.com/datasets/bobbyscience/league-of-legends-diamond-ranked-games-10-min)), coletado via Riot API. Contém estatísticas dos primeiros 10 minutos de partidas ranqueadas (solo queue) em elo alto (Diamond I a Master).

**Tarefa de classificação:** prever se o time azul vence a partida (`blueWins`: 1 = vitória azul, 0 = vitória vermelha), usando métricas in-game (kills, gold, wards, objetivos, etc.).

Base alternativa às listadas no enunciado, adequada a classificação supervisionada binária.

### 2) **(10%)** Pré-processamento: entendimento do conjunto de dados
- Quais são minhas features?
- Quais são minhas classes?
- Como estão distribuidas minhas classes?
- Checagem se os valores estão dentro de um limite permitido ou razoável.
- Tratamento de valores ausentes por eliminação ou substituição.
- Conversão do tipo de dados.


In [ ]:
import warnings
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Notebook autocontido (Google Colab ou local): baixa o CSV se não existir no diretório.
CSV_URL = (
    "https://github.com/JonnyLeeHJ/league-of-legends-outcome-classification/"
    "raw/refs/heads/master/data/high_diamond_ranked_10min.csv"
)
CSV_NAME = "high_diamond_ranked_10min.csv"

DATA_PATH = Path(CSV_NAME)
if not DATA_PATH.is_file():
    print("Baixando dataset (espelho público do Kaggle)...")
    urllib.request.urlretrieve(CSV_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(f"Dataset carregado: {DATA_PATH.name}")
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()



In [ ]:
# --- Features e classes ---
TARGET = "blueWins"
ID_COL = "gameId"

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET, ID_COL])

print("Classes (target):")
print(y.value_counts())
print("\nProporção:")
print(y.value_counts(normalize=True).round(4))

class_labels = {0: "Vitória vermelha", 1: "Vitória azul"}
ax = y.map(class_labels).value_counts().plot(kind="bar", color=["#c0392b", "#2980b9"], edgecolor="black")
ax.set_title("Distribuição das classes (blueWins)")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"\nNúmero de features: {X.shape[1]}")
print("Features:", list(X.columns))



In [ ]:
# --- Valores ausentes e tipos ---
print("Valores ausentes por coluna:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "Nenhum valor ausente.")

print("\nTipos de dados:")
print(df.dtypes)

# Conversão explícita do target
df[TARGET] = df[TARGET].astype(int)

# Checagem de limites razoáveis (contagens não negativas onde aplicável)
count_cols = [c for c in X.columns if any(k in c for k in ("Kills", "Deaths", "Gold", "Wards", "Minions", "Dragons", "Towers"))]
invalid = {c: int((X[c] < 0).sum()) for c in count_cols if (X[c] < 0).any()}
print("\nColunas com valores negativos (inesperados):", invalid if invalid else "Nenhuma.")

df.describe().T.head(12)



In [ ]:
# --- Distribuição de algumas features principais ---
sample_feats = [
    "blueGoldDiff", "blueKills", "blueTotalGold",
    "redGoldDiff", "redKills", "blueWardsPlaced",
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.ravel(), sample_feats):
    sns.histplot(X[col], kde=True, ax=ax, color="#34495e")
    ax.set_title(col)
plt.suptitle("Histogramas de features selecionadas", y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 10))
corr_df = X[sample_feats].copy()
corr_df[TARGET] = y
corr = corr_df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlação (features selecionadas + target)")
plt.tight_layout()
plt.show()



In [ ]:
# Divisão treino / teste (estratificada)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}")

results = []


def evaluate_classifier(name, model, X_tr, X_te, y_tr, y_te, plot_cm=True):
    """Métricas consistentes + comparação treino vs teste (overfitting)."""
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    metrics = {
        "modelo": name,
        "acc_treino": accuracy_score(y_tr, y_pred_tr),
        "acc_teste": accuracy_score(y_te, y_pred_te),
        "prec_teste": precision_score(y_te, y_pred_te, zero_division=0),
        "rec_teste": recall_score(y_te, y_pred_te, zero_division=0),
        "f1_teste": f1_score(y_te, y_pred_te, zero_division=0),
    }
    results.append(metrics)

    print(f"\n=== {name} ===")
    print(f"Acurácia treino: {metrics['acc_treino']:.4f} | teste: {metrics['acc_teste']:.4f}")
    print(classification_report(y_te, y_pred_te, target_names=["Vermelho", "Azul"]))

    if plot_cm:
        cm = confusion_matrix(y_te, y_pred_te)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=["Pred 0", "Pred 1"], yticklabels=["Real 0", "Real 1"])
        ax.set_title(f"Matriz de confusão — {name}")
        plt.tight_layout()
        plt.show()

    gap = metrics["acc_treino"] - metrics["acc_teste"]
    if gap > 0.08:
        print(f"Aviso: possivel overfitting (gap acuracia treino-teste = {gap:.4f})")
    elif metrics["acc_treino"] < metrics["acc_teste"] - 0.02:
        print("Aviso: acuracia de treino baixa — possivel underfitting.")
    else:
        print(f"Gap treino-teste (acurácia): {gap:.4f} — aceitável.")

    return model, metrics


def plot_train_test_bars(name, acc_tr, acc_te):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Treino", "Teste"], [acc_tr, acc_te], color=["#27ae60", "#e67e22"])
    ax.set_ylim(0, 1)
    ax.set_title(f"Acurácia treino vs teste — {name}")
    ax.set_ylabel("Acurácia")
    for i, v in enumerate([acc_tr, acc_te]):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
    plt.tight_layout()
    plt.show()



### 3) **(80%)** Nos blocos seguintes implemente seus classificadores (serão implementados 2 métodos diferentes).

#### 3.1) Qual método escolhido?

**Regressão Logística** — classificador linear probabilístico (baseline simples para classes binárias).

#### 3.2) **(10%)** Baseline - Implemente seu classificador da forma mais simples possível para esse ser seu baseline

In [ ]:
lr_baseline = LogisticRegression(max_iter=1000, random_state=42)
lr_baseline, m_lr_base = evaluate_classifier(
    "LR Baseline", lr_baseline, X_train, X_test, y_train, y_test
)
plot_train_test_bars("LR Baseline", m_lr_base["acc_treino"], m_lr_base["acc_teste"])



#### 3.3) **(20%)** Versão 1 - O que podemos fazer para melhorar nosso baseline? Aplique técnicas como redução de dimensionalidade, normalização ou outras. Compare os resultados.

In [ ]:
lr_pipe_v1 = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("clf", LogisticRegression(max_iter=2000, random_state=42)),
])

lr_pipe_v1, m_lr_v1 = evaluate_classifier(
    "LR v1 (Scaler + PCA 95%)", lr_pipe_v1, X_train, X_test, y_train, y_test
)
plot_train_test_bars("LR v1", m_lr_v1["acc_treino"], m_lr_v1["acc_teste"])

# Variância explicada pelo PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
pca_full = PCA().fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = int(np.searchsorted(cumvar, 0.95) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o", markersize=3)
ax.axhline(0.95, color="red", linestyle="--", label="95% variância")
ax.axvline(n_95, color="green", linestyle="--", label=f"n_components={n_95}")
ax.set_xlabel("Número de componentes")
ax.set_ylabel("Variância explicada acumulada")
ax.set_title("PCA — variância explicada acumulada")
ax.legend()
plt.tight_layout()
plt.show()

cmp = pd.DataFrame([m_lr_base, m_lr_v1])
cmp



#### 3.4) **(10%)** Tunning - Agora que temos um resultado promissor, vamos tentar melhorar o resultado alterando um ou mais hiper-parametro. Compare os resultados.

In [ ]:
param_grid_lr = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__solver": ["lbfgs", "liblinear"],
}

grid_lr = GridSearchCV(
    lr_pipe_v1,
    param_grid_lr,
    cv=5,
    scoring="f1",
    n_jobs=1,
)
grid_lr.fit(X_train, y_train)
print("Melhores hiperparâmetros (LR):", grid_lr.best_params_)
print(f"Melhor F1 (CV): {grid_lr.best_score_:.4f}")

lr_tuned = grid_lr.best_estimator_
lr_tuned, m_lr_tuned = evaluate_classifier(
    "LR Tunado (GridSearchCV)", lr_tuned, X_train, X_test, y_train, y_test, plot_cm=True
)
plot_train_test_bars("LR Tunado", m_lr_tuned["acc_treino"], m_lr_tuned["acc_teste"])

pd.DataFrame([m_lr_base, m_lr_v1, m_lr_tuned])



#### 3.5) Qual método escolhido?

**Random Forest** — ensemble de árvores de decisão, capaz de capturar relações não lineares entre features.

#### 3.6) **(10%)** Baseline - Implemente seu classificador da forma mais simples possível para esse ser seu baseline

In [ ]:
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf_baseline, m_rf_base = evaluate_classifier(
    "RF Baseline", rf_baseline, X_train, X_test, y_train, y_test
)
plot_train_test_bars("RF Baseline", m_rf_base["acc_treino"], m_rf_base["acc_teste"])



#### 3.7) **(20%)** Versão 1 - O que podemos fazer para melhorar nosso baseline? Aplique técnicas como redução de dimensionalidade, normalização ou outras. Compare os resultados.

In [ ]:
rf_pipe_v1 = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)),
])

rf_pipe_v1, m_rf_v1 = evaluate_classifier(
    "RF v1 (Scaler + PCA 95%)", rf_pipe_v1, X_train, X_test, y_train, y_test
)
plot_train_test_bars("RF v1", m_rf_v1["acc_treino"], m_rf_v1["acc_teste"])

pd.DataFrame([m_rf_base, m_rf_v1])



#### 3.8) **(10%)** Tunning - Agora que temos um resultado promissor, vamos tentar melhorar o resultado alterando um ou mais hiper-parametro. Compare os resultados.

In [ ]:
param_grid_rf = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [8, 12, None],
    "clf__min_samples_leaf": [1, 3, 5],
}

grid_rf = GridSearchCV(
    rf_pipe_v1,
    param_grid_rf,
    cv=3,
    scoring="f1",
    n_jobs=1,
)
grid_rf.fit(X_train, y_train)
print("Melhores hiperparâmetros (RF):", grid_rf.best_params_)
print(f"Melhor F1 (CV): {grid_rf.best_score_:.4f}")

rf_tuned = grid_rf.best_estimator_
rf_tuned, m_rf_tuned = evaluate_classifier(
    "RF Tunado (GridSearchCV)", rf_tuned, X_train, X_test, y_train, y_test
)
plot_train_test_bars("RF Tunado", m_rf_tuned["acc_treino"], m_rf_tuned["acc_teste"])

# Importância no RF baseline (features originais, mais interpretável)
rf_interp = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=1)
rf_interp.fit(X_train, y_train)
imp = pd.Series(rf_interp.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
imp.plot(kind="barh", ax=ax, color="#8e44ad")
ax.set_title("Top 15 importâncias de features — RF (dados originais)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

pd.DataFrame([m_rf_base, m_rf_v1, m_rf_tuned])



In [ ]:
summary = pd.DataFrame(results)
summary = summary.sort_values("f1_teste", ascending=False)
print(summary.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(summary))
w = 0.35
ax.bar(x - w/2, summary["acc_treino"], width=w, label="Acurácia treino", color="#3498db")
ax.bar(x + w/2, summary["acc_teste"], width=w, label="Acurácia teste", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(summary["modelo"], rotation=45, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("Comparação de todos os experimentos")
ax.legend()
plt.tight_layout()
plt.show()

# Curva ROC — melhor LR e melhor RF no teste
for label, est in [("LR Tunado", lr_tuned), ("RF Tunado", rf_tuned)]:
    if hasattr(est, "predict_proba"):
        proba = est.predict_proba(X_test)[:, 1]
    else:
        proba = est.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Aleatório")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curvas ROC — modelos tunados")
plt.legend()
plt.tight_layout()
plt.show()



### 4) **(10%)** Conclusões

**Comparação geral:** ambos os métodos superam um classificador ingênuo (~50% de acurácia) e atingem desempenho semelhante no conjunto de teste (acurácia e F1 na faixa de ~72–75%, típica para predição aos 10 minutos neste dataset). A Regressão Logística, após normalização e PCA, tende a ser mais estável e com menor gap treino–teste; o Random Forest pode alcançar acurácia ligeiramente maior no teste, porém com maior risco de overfitting no treino.

**Escolha para produção:** priorizaria a **Regressão Logística tunada** (com `StandardScaler` + PCA), por ser mais leve, interpretável (coeficientes por componente/feature) e mais fácil de monitorar em produção. Se a métrica de negócio exigir maximizar acurácia pura e houver infraestrutura para retreino frequente, o **Random Forest tunado** seria alternativa viável.

**Riscos:** (1) *concept drift* — patches e meta do LoL mudam a relação entre estatísticas e vitória; (2) uso indevido (apostas/harassment); (3) vazamento de informação se features posteriores aos 10 min forem incluídas; (4) correlação entre features azul/vermelho pode inflar importâncias sem causalidade.

**Próximos passos:** validação temporal (split por data), engenharia de features (apenas diffs azul−vermelho), calibração de probabilidades, dados mais recentes (ex. Season 15), e explicabilidade (SHAP) em features originais.

# Bônus

### 6) Deep Learning: O dilema da "Caixa-Preta" (Desempenho vs. Explicabilidade)

**Enunciado:**
Modelos de Deep Learning (Aprendizado Profundo) utilizam redes neurais artificiais com múltiplas camadas ocultas capazes de identificar padrões extremamente complexos em grandes volumes de dados — superando, muitas vezes, a precisão humana em tarefas como diagnósticos médicos por imagem ou análise de crédito. No entanto, quanto mais profunda e precisa é uma rede neural, menor costuma ser a sua interpretabilidade (o chamado problema da "caixa-preta"), tornando difícil explicar exatamente como o modelo chegou a uma determinada decisão.

**Pergunta:**
Considerando o funcionamento das redes neurais profundas, explique por que ocorre essa dificuldade em rastrear o "raciocínio" do modelo e posicione-se: em áreas críticas como saúde, justiça ou concessão de crédito, devemos priorizar modelos de Deep Learning com maior taxa de acerto (mesmo sem saber explicar suas decisões) ou modelos mais simples e transparentes, porém menos precisos? Justifique sua opinião propondo pelo menos um critério ou limite para o uso dessas redes na sociedade.

**Resposta (questão 6):**

Em redes profundas, milhões de parâmetros são ajustados por camadas: cada neurônio aplica uma combinação linear seguida de não linearidade, e o efeito final de uma decisão é distribuído por **centenas de interações** entre features intermediárias. Não existe, na prática, uma única regra legível do tipo “se gold diff > X, então vitória”; o que temos é um mapeamento de alta dimensão aprendido por otimização (gradiente), difícil de traduzir em linguagem natural para cada predição individual — daí o termo *caixa-preta*. Técnicas como LIME ou SHAP ajudam, mas são aproximações locais, não o “raciocínio” completo da rede.

**Posicionamento:** em saúde, justiça e crédito, eu **não** priorizaria apenas a maior acurácia de um deep learning opaco. O custo de um erro injustificado ou sem recurso de contestação é alto (dano à saúde, prisão indevida, negação de crédito discriminatória). Prefiro modelos **mais transparentes** (árvores, regressão logística, regras) quando a explicação faz parte do direito do afetado — mesmo aceitando alguma perda de desempenho — ou exijo que o DL seja acompanhado de auditoria e explicabilidade validada.

**Critério/limite sugerido:** usar redes profundas de forma ampla em decisões que **alteram direitos fundamentais ou risco de vida** somente se (1) houver ganho mensurável e reproduzível sobre alternativas interpretáveis, (2) existir **direito de revisão humana** e explicação em linguagem acessível, e (3) o modelo passar por avaliação de viés e monitoramento contínuo em produção. Fora dessas condições, o limite ético-legal deve favorecer transparência em detrimento de alguns pontos percentuais de acurácia.

### 7) IA Generativa: Criação, Autoria e os Limites da "Compreensão"

**Enunciado:**
Diferente da IA tradicional — voltada a classificar dados ou fazer previsões —, a IA Generativa (como Grandes Modelos de Linguagem e geradores de imagem) é capaz de produzir textos inéditos, códigos de programação, músicas e obras visuais a partir de padrões estatísticos aprendidos em bases de dados gigantescas. Por outro lado, esses modelos não possuem consciência ou vivência de mundo: eles operam prevendo o próximo elemento mais provável (tokens ou pixels) e estão sujeitos a fenômenos como as "alucinações" (geração de informações falsas com aparência convincente).

**Pergunta:**
A partir do modo como os modelos generativos são treinados e geram conteúdo, você considera que a IA Generativa é realmente capaz de ser "criativa" ou ela realiza apenas uma recombinação sofisticada do conhecimento humano já existente? Em sua resposta, explique como o funcionamento probabilístico desses modelos sustenta o seu ponto de vista e discuta como isso deve impactar o trabalho intelectual e artístico humano nos próximos anos.

**Resposta (questão 7):**

Do ponto de vista técnico, modelos generativos (LLMs, difusão em imagens etc.) estimam **P(próximo token | contexto)** ou distribuições sobre pixels/áudio a partir de bilhões de exemplos humanos. A “criação” é, portanto, **amostragem estocástica** sobre padrões já vistos: recombinação, extrapolação e interpolação sofisticadas, mas sem intenção, experiência corporal ou compreensão factual do mundo. Isso explica alucinações: o modelo otimiza plausibilidade linguística/estética, não verdade — pode inventar citações ou fatos porque a sequência *parece* correta estatisticamente.

Assim, eu não trataria a IA generativa como criatividade humana no sentido pleno (intencionalidade, contexto cultural vivido), mas como **ferramenta de recombinação acelerada** que amplia o repertório humano — semelhante a um sintetizador ou um collage digital, porém em escala massiva.

**Impacto nos próximos anos:** o trabalho intelectual e artístico tende a se reorganizar: rotinas repetitivas (rascunhos, variações, boilerplate de código) serão delegadas; o valor humano migrará para **curadoria, direção criativa, verificação de fatos, ética e originalidade conceitual**. Profissões precisarão declarar uso de IA, proteger autoria e remunerar dados de treino; artistas e pesquisadores que souberem **editar e criticar** o output da máquina (em vez de apenas publicá-lo) ganharão diferencial. Em resumo: a IA generativa não substitui a criatividade humana, mas **redesenha** quem faz o quê — exigindo mais literacia crítica sobre o que é provável, não necessariamente verdadeiro ou único.